In [2]:
#  ___________________________________________________________________________
#
#  Pyomo: Python Optimization Modeling Objects
#  Copyright (c) 2008-2025
#  National Technology and Engineering Solutions of Sandia, LLC
#  Under the terms of Contract DE-NA0003525 with National Technology and
#  Engineering Solutions of Sandia, LLC, the U.S. Government retains certain
#  rights in this software.
#  This software is distributed under the 3-clause BSD License.
#  ___________________________________________________________________________

# === Required imports ===
import os
import sys
import shutil
from pathlib import Path

import pyomo.environ as pyo
from pyomo.dae import ContinuousSet, DerivativeVar
from pyomo.contrib.parmest.experiment import Experiment
from pyomo.common.dependencies import numpy as np

from pyomo.contrib.doe import DesignOfExperiments

# --------------------
# IPOPT: explicit setup (prevents "Solver (IPOPT) not available")
# --------------------
IPOPT_BIN = shutil.which("ipopt")
assert IPOPT_BIN, "ipopt executable not found on PATH (activate the 'doe' env)."
ipopt_solver = pyo.SolverFactory("ipopt", executable=IPOPT_BIN)

# --------------------
# Excel helpers (single workbook, single sheet)
# --------------------
from openpyxl import Workbook, load_workbook

def _ensure_book_and_sheet(xlsx_path: Path, sheet_name: str,
                           header_cols: list[str], notebook_id: str, scenario: str):
    """
    Ensure outputs/metrics.xlsx exists and has a sheet `sheet_name`.
    On first creation of that sheet, write NOTEBOOK_ID & scenario ONCE (row 1),
    then a blank row, then the header row.
    """
    xlsx_path.parent.mkdir(parents=True, exist_ok=True)
    if xlsx_path.exists():
        wb = load_workbook(xlsx_path)
    else:
        wb = Workbook()
        # remove default empty sheet if it's the only one
        if wb.sheetnames == ["Sheet"]:
            wb.remove(wb["Sheet"])

    if sheet_name in wb.sheetnames:
        ws = wb[sheet_name]
        # if the sheet is empty, add the header block
        if ws.max_row == 1 and all(c.value is None for c in ws[1]):
            ws.append(["NOTEBOOK_ID", notebook_id, "SCENARIO", scenario])
            ws.append([])  # blank line
            ws.append(header_cols)
    else:
        ws = wb.create_sheet(title=sheet_name[:31])  # Excel sheet name <= 31 chars
        ws.append(["NOTEBOOK_ID", notebook_id, "SCENARIO", scenario])
        ws.append([])  # blank line
        ws.append(header_cols)

    return wb, ws

def _append_row(ws, row):
    ws.append(row)

def _save_wb(wb, xlsx_path: Path):
    wb.save(xlsx_path)

# --------------------
# Experiment definition
# --------------------
class ReactorExperiment(Experiment):
    def __init__(self, data, nfe, ncp):
        self.data = data
        self.nfe = nfe
        self.ncp = ncp
        self.model = None

    def get_labeled_model(self):
        if self.model is None:
            self.create_model()
            self.finalize_model()
            self.label_experiment()
        return self.model

    def create_model(self):
        m = self.model = pyo.ConcreteModel()

        # Parameters
        m.R = pyo.Param(mutable=False, initialize=8.314)

        # Time
        m.t = ContinuousSet(bounds=[0, 1])

        # States
        m.CA = pyo.Var(m.t, within=pyo.NonNegativeReals)
        m.CB = pyo.Var(m.t, within=pyo.NonNegativeReals)
        m.CC = pyo.Var(m.t, within=pyo.NonNegativeReals)
        m.T = pyo.Var(m.t, within=pyo.NonNegativeReals)

        # Unknown parameters
        m.A1 = pyo.Var(within=pyo.NonNegativeReals)
        m.E1 = pyo.Var(within=pyo.NonNegativeReals)
        m.A2 = pyo.Var(within=pyo.NonNegativeReals)
        m.E2 = pyo.Var(within=pyo.NonNegativeReals)

        # Derivatives
        m.dCAdt = DerivativeVar(m.CA, wrt=m.t)
        m.dCBdt = DerivativeVar(m.CB, wrt=m.t)

        # Kinetics
        @m.Expression(m.t)
        def k1(m, t):
            return m.A1 * pyo.exp(-m.E1 * 1000 / (m.R * m.T[t]))

        @m.Expression(m.t)
        def k2(m, t):
            return m.A2 * pyo.exp(-m.E2 * 1000 / (m.R * m.T[t]))

        # ODEs
        @m.Constraint(m.t)
        def CA_rxn_ode(m, t):
            return m.dCAdt[t] == -m.k1[t] * m.CA[t]

        @m.Constraint(m.t)
        def CB_rxn_ode(m, t):
            return m.dCBdt[t] == m.k1[t] * m.CA[t] - m.k2[t] * m.CB[t]

        # Algebraic balance for C (equimolar A->B->C)
        @m.Constraint(m.t)
        def CC_balance(m, t):
            return m.CA[0] == m.CA[t] + m.CB[t] + m.CC[t]

    def finalize_model(self):
        m = self.model
        control_points = self.data["control_points"]

        # Initial conditions
        m.CA[0].value = self.data["CA0"]
        m.CB[0].fix(self.data["CB0"])

        # Time grid & control points
        m.t.update(self.data["t_range"])
        m.t.update(control_points)

        # Fix unknown parameters
        m.A1.fix(self.data["A1"])
        m.A2.fix(self.data["A2"])
        m.E1.fix(self.data["E1"])
        m.E2.fix(self.data["E2"])

        # Bounds on design variable and temperature
        m.CA[0].setlb(self.data["CA_bounds"][0])
        m.CA[0].setub(self.data["CA_bounds"][1])
        m.t_control = control_points

        discr = pyo.TransformationFactory("dae.collocation")
        discr.apply_to(m, nfe=self.nfe, ncp=self.ncp, wrt=m.t)

        # Initialize & bound temperature, piecewise-constant between control points
        cv = None
        for t in m.t:
            if t in control_points:
                cv = control_points[t]
                m.T[t].fix()
            m.T[t].setlb(self.data["T_bounds"][0])
            m.T[t].setub(self.data["T_bounds"][1])
            m.T[t] = cv

        @m.Constraint(m.t - control_points)
        def T_control(m, t):
            neighbour_t = max(tc for tc in control_points if tc < t)
            return m.T[t] == m.T[neighbour_t]

    def label_experiment(self):
        m = self.model
        m.experiment_outputs = pyo.Suffix(direction=pyo.Suffix.LOCAL)
        m.experiment_outputs.update((m.CA[t], None) for t in m.t_control)
        m.experiment_outputs.update((m.CB[t], None) for t in m.t_control)
        m.experiment_outputs.update((m.CC[t], None) for t in m.t_control)

        m.measurement_error = pyo.Suffix(direction=pyo.Suffix.LOCAL)
        concentration_error = 1e-2
        m.measurement_error.update((m.CA[t], concentration_error) for t in m.t_control)
        m.measurement_error.update((m.CB[t], concentration_error) for t in m.t_control)
        m.measurement_error.update((m.CC[t], concentration_error) for t in m.t_control)

        m.experiment_inputs = pyo.Suffix(direction=pyo.Suffix.LOCAL)
        m.experiment_inputs[m.CA[m.t.first()]] = None
        m.experiment_inputs.update((m.T[t], None) for t in m.t_control)

        m.unknown_parameters = pyo.Suffix(direction=pyo.Suffix.LOCAL)
        m.unknown_parameters.update((k, pyo.value(k)) for k in [m.A1, m.A2, m.E1, m.E2])

# --------------------
# Static data + options
# --------------------
data_ex = {
    "CA0": 5.0, "CA_bounds": [1.0, 5.0], "CB0": 0.0, "CC0": 0.0, "t_range": [0, 1],
    "control_points": {"0": 500, "0.125": 300, "0.25": 300, "0.375": 300, "0.5": 300,
                       "0.625": 300, "0.75": 300, "0.875": 300, "1": 300},
    "T_bounds": [300, 700], "A1": 84.79, "A2": 371.72, "E1": 7.78, "E2": 15.05
}
data_ex["control_points"] = {float(k): v for k, v in data_ex["control_points"].items()}

fd_formula = "central"
step_size = 1e-3
objective_option = "determinant"
scale_nominal_param_value = True

# --------------------
# Notebook / Scenario / Excel config
# --------------------
NOTEBOOK_ID = "4"
scenario = "central"
XLSX_PATH = Path("outputs/metrics.xlsx")
SHEET_NAME = f"{NOTEBOOK_ID}_{scenario}"[:31]  # Excel sheet name limit

# Header for the table rows we append
ROW_HEADER = [
    "run", "solve_time_s", "build_time_s", "init_time_s", "total_wall_time_s", "optimal_experiment_values"
]

# Ensure workbook & sheet with NOTEBOOK_ID+scenario written once
_wb, _ws = _ensure_book_and_sheet(XLSX_PATH, SHEET_NAME, ROW_HEADER, NOTEBOOK_ID, scenario)
_save_wb(_wb, XLSX_PATH)

# --------------------
# A single run (build fresh objects, run, return metrics & opt values)
# --------------------
def run_single():
    experiment = ReactorExperiment(data=data_ex, nfe=10, ncp=3)
    doe_obj = DesignOfExperiments(
        experiment,
        fd_formula=fd_formula,
        step=step_size,
        objective_option=objective_option,
        scale_constant_value=1,
        scale_nominal_param_value=scale_nominal_param_value,
        prior_FIM=None,
        jac_initial=None,
        fim_initial=None,
        L_diagonal_lower_bound=1e-7,
        solver=ipopt_solver,
        tee=False,
        get_labeled_model_args=None,
        _Cholesky_option=True,
        _only_compute_fim_lower=True,
    )
    # Optional: you could compute factorial here if desired
    doe_obj.run_doe()

    # Collect metrics
    solve_time = doe_obj.results["Solve Time"]
    build_time = doe_obj.results["Build Time"]
    init_time  = doe_obj.results["Initialization Time"]
    total_time = doe_obj.results["Wall-clock Time"]

    # Save optimal experimental values (as a compact string)
    # doe_obj.results["Experiment Design"] is a list-like; convert to CSV string
    design_vals = doe_obj.results["Experiment Design"]
    design_vals_str = ",".join(f"{v:.6g}" for v in design_vals)

    return solve_time, build_time, init_time, total_time, design_vals_str

# --------------------
# Run 100 times, append rows, print the 100th result
# --------------------
results_100 = None

# open workbook once, append rows, save once at end (safer/faster)
wb = load_workbook(XLSX_PATH) if XLSX_PATH.exists() else Workbook()
if SHEET_NAME in wb.sheetnames:
    ws = wb[SHEET_NAME]
else:
    # create and add headers if somehow missing
    ws = wb.create_sheet(title=SHEET_NAME)
    ws.append(["NOTEBOOK_ID", NOTEBOOK_ID, "SCENARIO", scenario])
    ws.append([])
    ws.append(ROW_HEADER)

for run in range(1, 101):
    print(run)
    solve_time, build_time, init_time, total_time, design_vals_str = run_single()
    _append_row(ws, [run, solve_time, build_time, init_time, total_time, design_vals_str])
    if run == 100:
        results_100 = (run, solve_time, build_time, init_time, total_time, design_vals_str)

_save_wb(wb, XLSX_PATH)

# --------------------
# Print ONLY the 100th run
# --------------------
if results_100 is not None:
    r, st, bt, it, tt, dv = results_100
    print(f"100th run -> run={r}, solve_time_s={st}, build_time_s={bt}, init_time_s={it}, total_wall_time_s={tt}")
    print(f"Optimal experimental values (run 100): {dv}")


1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
100th run -> run=100, solve_time_s=0.15218583308160305, build_time_s=0.24397800001315773, init_time_s=0.12019812490325421, total_wall_time_s=0.516361957998015
Optimal experimental values (run 100): 5,481.88,300,300,300,300,300,300,300,300.001
